In [1]:
import h5py
import scipy.io as io
import PIL.Image as Image
import numpy as np
import os
import glob
from matplotlib import pyplot as plt
from scipy.ndimage.filters import gaussian_filter 
import scipy
import json
import torchvision.transforms.functional as F
from matplotlib import cm as CM
from image import *
from model import CSRNet
import torch
%matplotlib inline

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_33028\1028975186.py:8: DeprecationWarning: Please use `gaussian_filter` from the `scipy.ndimage` namespace, the `scipy.ndimage.filters` namespace is deprecated.
  from scipy.ndimage.filters import gaussian_filter


In [2]:
from torchvision import datasets, transforms
transform=transforms.Compose([
                       transforms.ToTensor(),transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225]),
                   ])

In [3]:
root = 'D:/SauDaiHoc/Deep Learning/FinalProject/Shanghai/'

In [4]:
#now generate the ShanghaiA's ground truth
part_A_train = os.path.join(root,'part_A_final/train_data','images')
part_A_test = os.path.join(root,'part_A_final/test_data','images')
part_B_train = os.path.join(root,'part_B_final/train_data','images')
part_B_test = os.path.join(root,'part_B_final/test_data','images')
path_sets = [part_A_test]

In [5]:
img_paths = []
for path in path_sets:
    for img_path in glob.glob(os.path.join(path, '*.jpg')):
        img_paths.append(img_path)

In [6]:
model = CSRNet()

In [7]:
model = model.cpu()

In [10]:
checkpoint = torch.load('D:\SauDaiHoc\Deep Learning\FinalProject\PartAmodel_best.pth.tar',
                        map_location=torch.device('cpu'))

In [11]:
model.load_state_dict(checkpoint['state_dict'])

<All keys matched successfully>

In [13]:
mae = 0
for i in range(len(img_paths)):
    img = 255.0 * F.to_tensor(Image.open(img_paths[i]).convert('RGB'))

    img[0,:,:]=img[0,:,:]-92.8207477031
    img[1,:,:]=img[1,:,:]-95.2757037428
    img[2,:,:]=img[2,:,:]-104.877445883
    img = img.cpu()
    #img = transform(Image.open(img_paths[i]).convert('RGB')).cuda()
    gt_file = h5py.File(img_paths[i].replace('.jpg','.h5').replace('images','ground_truth'),'r')
    groundtruth = np.asarray(gt_file['density'])
    output = model(img.unsqueeze(0))
    mae += abs(output.detach().cpu().sum().numpy()-np.sum(groundtruth))
    print(i, mae)
print(mae/len(img_paths))

d:\SauDaiHoc\Deep Learning\FinalProject\csrnet_env\lib\site-packages\torch\nn\functional.py:718: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at  ..\c10/core/TensorImpl.h:1156.)
  return torch.max_pool2d(input, kernel_size, stride, padding, dilation, ceil_mode)


0 2294.58349609375
1 5220.867431640625
2 6670.71923828125
3 8488.279541015625
4 11490.422607421875
5 14223.5771484375
6 19216.958984375
7 22212.41552734375
8 36664.00341796875
9 37379.94104003906
10 39358.37414550781
11 44104.23205566406
12 51608.20422363281
13 60751.26477050781
14 61933.05517578125
15 65827.53735351562
16 66461.44219970703
17 67127.9340209961
18 75317.67669677734
19 76460.1664428711
20 88992.8637084961
21 90685.3080444336
22 93349.91766357422
23 93531.65930175781
24 94421.23266601562
25 104119.03735351562
26 111098.93579101562
27 112748.71350097656
28 114336.00537109375
29 117398.94116210938
30 120743.73291015625
31 124838.4404296875
32 126817.16784667969
33 127015.8362121582
34 135410.0432434082
35 137803.8742980957
36 144150.0637512207
37 146829.86892700195
38 148222.80435180664
39 149032.71939086914
40 149360.6792602539
41 149967.14764404297
42 155514.09442138672
43 158655.52508544922
44 160854.39349365234
45 167256.07708740234
46 168322.3920288086
47 172860.390563